# Digital Net Base 2 (Sobol' Sequence) Demo

Translated from QMCJu's digital_net_b2.ipynb

Demonstrates Sobol' point generation with various scrambling methods,
replications, and integration.

In [ ]:
using QMCJu
using Statistics
using Printf

## Basic Usage

In [ ]:
println("="^60)
println("Basic Sobol' Point Generation")
println("="^60)

dn = DigitalNetB2(5; seed=7)
println("  $dn")
println()

x = gen_samples(dn, 4)
println("  4 points in 5 dimensions:")
for i in 1:4
    @printf("    [%.4f  %.4f  %.4f  %.4f  %.4f]\n", x[i,:]...)
end
println()

## Larger sample

In [ ]:
x16 = gen_samples(dn, 16)
println("  16 points, first 3 dims (first 8 shown):")
for i in 1:8
    @printf("    [%.4f  %.4f  %.4f]\n", x16[i,1], x16[i,2], x16[i,3])
end
println("    ...")
println()

## Randomization Methods

In [ ]:
println("="^60)
println("Randomization Methods")
println("="^60)

for method in ["none", "DS", "LMS_DS"]
    dn_m = DigitalNetB2(3; randomize=method, seed=42)
    x_m = gen_samples(dn_m, 8)
    println("  Randomize = \"$method\":")
    for i in 1:4
        @printf("    [%.6f  %.6f  %.6f]\n", x_m[i,:]...)
    end
    println("    ...")
    println()
end

## Reproducibility with seeds

In [ ]:
println("="^60)
println("Reproducibility")
println("="^60)

dn_a = DigitalNetB2(3; seed=42)
dn_b = DigitalNetB2(3; seed=42)
xa = gen_samples(dn_a, 64)
xb = gen_samples(dn_b, 64)
println("  Same seed → identical points: $(xa ≈ xb)")

dn_c = DigitalNetB2(3; seed=99)
xc = gen_samples(dn_c, 64)
println("  Different seed → different:   $(!(xa ≈ xc))")
println()

## Replications

In [ ]:
println("="^60)
println("Replications (Independent Randomizations)")
println("="^60)

dn_r = DigitalNetB2(3; seed=7, replications=4)
xr = gen_samples(dn_r, 16)
println("  Shape: $(size(xr))  (R × n × d)")
println("  Replication 1, first 4 points:")
for i in 1:4
    @printf("    [%.4f  %.4f  %.4f]\n", xr[1,i,1], xr[1,i,2], xr[1,i,3])
end
println("  Replication 2, first 4 points:")
for i in 1:4
    @printf("    [%.4f  %.4f  %.4f]\n", xr[2,i,1], xr[2,i,2], xr[2,i,3])
end
println()

## Unscrambled (deterministic) Sobol' points

In [ ]:
println("="^60)
println("Deterministic Sobol' Points (no randomization)")
println("="^60)

dn_det = DigitalNetB2(2; randomize="none")
x_det = gen_samples(dn_det, 16)
println("  First 16 Sobol' points in 2D:")
for i in 1:16
    @printf("    %2d: [%.4f  %.4f]\n", i, x_det[i,1], x_det[i,2])
end
println()

## High-dimensional usage

In [ ]:
println("="^60)
println("High-Dimensional Usage")
println("="^60)

dn_hd = DigitalNetB2(52; seed=7)
xhd = gen_samples(dn_hd, 1024)
println("  52-dimensional Sobol' points: size = $(size(xhd))")
println("  Mean per dim (should be ≈ 0.5):")
@printf("    dim 1:  %.4f\n", mean(xhd[:, 1]))
@printf("    dim 26: %.4f\n", mean(xhd[:, 26]))
@printf("    dim 52: %.4f\n", mean(xhd[:, 52]))
println()

## Integration with Sobol'

In [ ]:
println("="^60)
println("Integration: ∫₀¹ ∫₀¹ (x₁ + x₂) dx = 1.0")
println("="^60)

dd = DigitalNetB2(2; randomize="LMS_DS", seed=7)
tm = Uniform(dd)
f = CustomFun(tm, x -> sum(x, dims=2)[:])
sc = CubQMCNetG(f; abs_tol=0.001, n_init=2^8, n_reps=16)
result = integrate(sc)
@printf("  Solution:  %.6f  (exact = 1.0)\n", result.solution)
@printf("  Error:     %.2e\n", abs(result.solution - 1.0))
@printf("  Samples:   %d per replication\n", result.data[:n])

println()
println("="^60)
println("Digital Net demo completed!")